# DPC-UNet Fine-Tuning on Adversarial Examples

Fine-tunes the DPC-UNet denoising defense on adversarial images generated by FGSM, PGD,
and DeepFool against YOLOv8. Optimizes for pixel fidelity **and edge preservation** so
the denoised output retains spatial features YOLO depends on for detection.

**Before running this notebook:**
1. Locally: `python scripts/export_training_data.py` → creates `outputs/dpc_unet_training.zip`
2. Upload `dpc_unet_training.zip` to your **Google Drive root**
3. In Colab: Runtime → Change runtime type → **T4 GPU**

**After training:**
- Download `dpc_unet_adversarial_finetuned.pt` from Drive
- Copy it to `YOLO-Bad-Triangle/`
- Set `DPC_UNET_CHECKPOINT_PATH=dpc_unet_adversarial_finetuned.pt` in `.env`
- Re-run the sweep to evaluate

In [ ]:
# Cell 1: Check GPU
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# Cell 2: Install missing dependencies (cv2 headless for Colab)
!pip install opencv-python-headless -q

In [ ]:
# Cell 3: Mount Google Drive and extract training data
import os, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ZIP_PATH    = '/content/drive/MyDrive/dpc_unet_training.zip'
EXTRACT_DIR = Path('/content/training_data')

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Upload dpc_unet_training.zip to your Google Drive root first.\n'
        f'Expected at: {ZIP_PATH}'
    )

if not EXTRACT_DIR.exists():
    print('Extracting training data...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print('Training data already extracted.')

clean_imgs    = sorted((EXTRACT_DIR / 'clean').glob('*.jpg'))
fgsm_imgs     = sorted((EXTRACT_DIR / 'adversarial/fgsm').glob('*.jpg'))
pgd_imgs      = sorted((EXTRACT_DIR / 'adversarial/pgd').glob('*.jpg'))
deepfool_imgs = sorted((EXTRACT_DIR / 'adversarial/deepfool').glob('*.jpg'))
checkpoint    = EXTRACT_DIR / 'checkpoint/dpc_unet_final_golden.pt'

print(f'Clean:    {len(clean_imgs)} images')
print(f'FGSM:     {len(fgsm_imgs)} images')
print(f'PGD:      {len(pgd_imgs)} images')
print(f'DeepFool: {len(deepfool_imgs)} images')
print(f'Checkpoint exists: {checkpoint.exists()}')

In [ ]:
# Cell 4: DPC-UNet architecture (exact match to production wrapper)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


def sinusoidal_timestep_embedding(timesteps: Tensor, dim: int = 64) -> Tensor:
    if timesteps.ndim == 0:
        timesteps = timesteps.unsqueeze(0)
    timesteps = timesteps.float()
    half = dim // 2
    if half == 0:
        return timesteps.unsqueeze(-1)
    device = timesteps.device
    exponent = torch.arange(half, device=device, dtype=torch.float32) / max(half - 1, 1)
    freq = torch.exp(-torch.log(torch.tensor(10000.0, device=device)) * exponent)
    args = timesteps.unsqueeze(-1) * freq.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


def _num_groups(channels: int) -> int:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class DPCResidualBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.mlp   = nn.Sequential(nn.SiLU(), nn.Linear(64, out_ch))
        self.conv1 = nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(_num_groups(out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(_num_groups(out_ch), out_ch)

    def forward(self, x: Tensor, t_embed: Tensor) -> Tensor:
        h = self.conv1(x)
        h = h + self.mlp(t_embed).unsqueeze(-1).unsqueeze(-1)
        h = F.silu(self.norm1(h))
        h = self.conv2(h)
        h = F.silu(self.norm2(h))
        return h


class DPCUNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.time_mlp   = nn.Sequential(nn.SiLU(), nn.Linear(64, 64))
        self.down1      = DPCResidualBlock(3, 32)
        self.down2      = DPCResidualBlock(32, 64)
        self.bottleneck = DPCResidualBlock(64, 128)
        self.up2_conv   = nn.Sequential(nn.Conv2d(128, 256, kernel_size=1), nn.PixelShuffle(2))
        self.up2_block  = DPCResidualBlock(128, 64)
        self.up1_conv   = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1), nn.PixelShuffle(2))
        self.up1_block  = DPCResidualBlock(64, 32)
        self.final      = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, x: Tensor, timestep=50.0) -> Tensor:
        B = x.shape[0]
        if isinstance(timestep, (int, float)):
            t = torch.full((B,), float(timestep), device=x.device)
        else:
            t = timestep.to(x.device)
        t_embed = self.time_mlp(sinusoidal_timestep_embedding(t, dim=64))

        x1 = self.down1(x, t_embed)
        x2 = self.down2(F.avg_pool2d(x1, 2), t_embed)
        x3 = self.bottleneck(F.avg_pool2d(x2, 2), t_embed)

        up2 = self.up2_conv(x3)
        if up2.shape[-2:] != x2.shape[-2:]:
            up2 = F.interpolate(up2, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        up2 = self.up2_block(torch.cat([up2, x2], dim=1), t_embed)

        up1 = self.up1_conv(up2)
        if up1.shape[-2:] != x1.shape[-2:]:
            up1 = F.interpolate(up1, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        up1 = self.up1_block(torch.cat([up1, x1], dim=1), t_embed)
        return self.final(up1)


print(f'DPCUNet params: {sum(p.numel() for p in DPCUNet().parameters()):,}')

In [ ]:
# Cell 5: Dataset — loads (adversarial, clean) pairs with random crops and flips
import random
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader


class AdvPairDataset(Dataset):
    """
    Each item is (adversarial_tensor, clean_tensor), both [3, H, W] in [0, 1] RGB.
    Pairs are built from every (clean_image, attack_type) combination.
    """

    def __init__(self, pairs: list, patch_size: int = 256, augment: bool = True):
        self.pairs      = pairs   # list of (clean_path_str, adv_path_str, attack_name)
        self.patch_size = patch_size
        self.augment    = augment

    @classmethod
    def from_dirs(cls, clean_dir: Path, adv_dirs: dict, patch_size: int = 256, augment: bool = True):
        clean_paths = sorted(Path(clean_dir).glob('*.jpg'))
        pairs = []
        for clean_path in clean_paths:
            for attack_name, adv_dir in adv_dirs.items():
                adv_path = Path(adv_dir) / clean_path.name
                if adv_path.exists():
                    pairs.append((str(clean_path), str(adv_path), attack_name))
        print(f'  {len(clean_paths)} images × {len(adv_dirs)} attacks = {len(pairs)} pairs')
        return cls(pairs, patch_size=patch_size, augment=augment)

    def split(self, val_frac: float = 0.15, seed: int = 42):
        pairs = self.pairs.copy()
        random.Random(seed).shuffle(pairs)
        n_val = max(1, int(len(pairs) * val_frac))
        val_ds   = AdvPairDataset(pairs[:n_val],  self.patch_size, augment=False)
        train_ds = AdvPairDataset(pairs[n_val:],  self.patch_size, augment=True)
        return train_ds, val_ds

    def __len__(self) -> int:
        return len(self.pairs)

    def _load_and_crop(self, clean_path: str, adv_path: str):
        clean = cv2.imread(clean_path)
        adv   = cv2.imread(adv_path)
        if clean is None or adv is None:
            raise RuntimeError(f'Failed to load image pair: {clean_path}')

        p = self.patch_size
        h, w = clean.shape[:2]

        # Pad if smaller than patch
        if h < p or w < p:
            ph, pw = max(0, p - h), max(0, p - w)
            clean = cv2.copyMakeBorder(clean, 0, ph, 0, pw, cv2.BORDER_REFLECT)
            adv   = cv2.copyMakeBorder(adv,   0, ph, 0, pw, cv2.BORDER_REFLECT)
            h, w  = clean.shape[:2]

        if self.augment:
            y = random.randint(0, h - p)
            x = random.randint(0, w - p)
        else:
            y = (h - p) // 2
            x = (w - p) // 2

        return clean[y:y+p, x:x+p], adv[y:y+p, x:x+p]

    @staticmethod
    def _to_tensor(img_bgr: np.ndarray) -> torch.Tensor:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return torch.from_numpy(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1)

    def __getitem__(self, idx: int):
        clean_path, adv_path, _ = self.pairs[idx]
        clean_bgr, adv_bgr = self._load_and_crop(clean_path, adv_path)

        if self.augment:
            if random.random() < 0.5:   # horizontal flip
                clean_bgr = cv2.flip(clean_bgr, 1)
                adv_bgr   = cv2.flip(adv_bgr,   1)
            if random.random() < 0.3:   # vertical flip
                clean_bgr = cv2.flip(clean_bgr, 0)
                adv_bgr   = cv2.flip(adv_bgr,   0)

        # input = adversarial, target = clean
        return self._to_tensor(adv_bgr), self._to_tensor(clean_bgr)

In [ ]:
# Cell 6: Loss functions
#
# L1 pixel loss:  forces pixel-level accuracy toward the clean image
# Sobel edge loss: forces edge/gradient preservation — directly targets the
#                  spatial features YOLO uses for detection

def sobel_edges(x: torch.Tensor) -> torch.Tensor:
    """Compute per-channel Sobel edge magnitude [B, C, H, W]."""
    kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                      dtype=torch.float32, device=x.device)
    ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                      dtype=torch.float32, device=x.device)
    C  = x.shape[1]
    kx = kx.view(1, 1, 3, 3).expand(C, 1, 3, 3)
    ky = ky.view(1, 1, 3, 3).expand(C, 1, 3, 3)
    gx = F.conv2d(x, kx, padding=1, groups=C)
    gy = F.conv2d(x, ky, padding=1, groups=C)
    return torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)


def denoising_loss(
    output: torch.Tensor,
    target: torch.Tensor,
    edge_weight: float = 0.15,
) -> tuple:
    """
    Returns (total_loss, pixel_loss_scalar, edge_loss_scalar).
    total = L1(output, target) + edge_weight * L1(edges(output), edges(target))
    """
    pixel = F.l1_loss(output, target)
    edge  = F.l1_loss(sobel_edges(output), sobel_edges(target))
    total = pixel + edge_weight * edge
    return total, pixel.item(), edge.item()

In [ ]:
# Cell 7: Hyperparameters

LEARNING_RATE  = 2e-5    # Small LR to fine-tune without destroying pre-trained features
BATCH_SIZE     = 8       # Fits comfortably on T4 (16 GB) with 256x256 patches
NUM_EPOCHS     = 60
PATCH_SIZE     = 256     # Random crop size
TIMESTEP_MIN   = 10.0    # Minimum timestep sampled during training
TIMESTEP_MAX   = 75.0    # Maximum timestep (matches inference range)
EDGE_WEIGHT    = 0.15    # Weight for Sobel edge loss component
VAL_FRAC       = 0.15    # Fraction of pairs held out for validation
GRAD_CLIP      = 1.0     # Gradient clipping max norm
SEED           = 42
NUM_WORKERS    = 2

SAVE_PATH      = '/content/drive/MyDrive/dpc_unet_adversarial_finetuned.pt'

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('Config set.')

In [ ]:
# Cell 8: Build dataset and dataloaders

ADV_DIRS = {
    'fgsm':     EXTRACT_DIR / 'adversarial/fgsm',
    'pgd':      EXTRACT_DIR / 'adversarial/pgd',
    'deepfool': EXTRACT_DIR / 'adversarial/deepfool',
}

print('Building dataset...')
full_ds = AdvPairDataset.from_dirs(EXTRACT_DIR / 'clean', ADV_DIRS,
                                   patch_size=PATCH_SIZE, augment=True)
train_ds, val_ds = full_ds.split(val_frac=VAL_FRAC, seed=SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):>5} pairs  ({len(train_loader)} batches/epoch)')
print(f'Val:   {len(val_ds):>5} pairs  ({len(val_loader)} batches)')

In [ ]:
# Cell 9: Load pre-trained checkpoint and set up optimizer

model = DPCUNet().to(device)
state_dict = torch.load(str(checkpoint), map_location='cpu', weights_only=True)
model.load_state_dict(state_dict, strict=True)
print(f'Loaded pre-trained checkpoint: {checkpoint.name}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)
print(f'Optimizer: Adam lr={LEARNING_RATE}, cosine decay to 1e-7 over {NUM_EPOCHS} epochs')

In [ ]:
# Cell 10: Training loop
from tqdm.notebook import tqdm

best_val_loss = float('inf')
history = {'train': [], 'val': [], 'pixel': [], 'edge': []}

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    ep_loss = ep_pixel = ep_edge = 0.0

    bar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{NUM_EPOCHS} train', leave=False)
    for adv_imgs, clean_imgs in bar:
        adv_imgs   = adv_imgs.to(device, non_blocking=True)
        clean_imgs = clean_imgs.to(device, non_blocking=True)

        # Random timestep per batch — covers the inference range
        t = random.uniform(TIMESTEP_MIN, TIMESTEP_MAX)

        optimizer.zero_grad()
        output = model(adv_imgs, timestep=t)
        output = torch.clamp(output, 0.0, 1.0)

        loss, pixel_l, edge_l = denoising_loss(output, clean_imgs, EDGE_WEIGHT)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        ep_loss  += loss.item()
        ep_pixel += pixel_l
        ep_edge  += edge_l
        bar.set_postfix(loss=f'{loss.item():.4f}', t=f'{t:.0f}')

    train_loss  = ep_loss  / len(train_loader)
    train_pixel = ep_pixel / len(train_loader)
    train_edge  = ep_edge  / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for adv_imgs, clean_imgs in val_loader:
            adv_imgs   = adv_imgs.to(device, non_blocking=True)
            clean_imgs = clean_imgs.to(device, non_blocking=True)
            output = torch.clamp(model(adv_imgs, timestep=50.0), 0.0, 1.0)
            loss, _, _ = denoising_loss(output, clean_imgs, EDGE_WEIGHT)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    scheduler.step()

    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['pixel'].append(train_pixel)
    history['edge'].append(train_edge)

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)

    lr_now = scheduler.get_last_lr()[0]
    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS}  '
        f'train={train_loss:.4f} (px={train_pixel:.4f} edge={train_edge:.4f})  '
        f'val={val_loss:.4f}  '
        f'lr={lr_now:.1e}'
        + ('  ← best' if is_best else '')
    )

print(f'\nTraining complete. Best val loss: {best_val_loss:.4f}')
print(f'Checkpoint saved to: {SAVE_PATH}')

In [ ]:
# Cell 11: Training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history['train']) + 1)
ax1.plot(epochs, history['train'], label='train')
ax1.plot(epochs, history['val'],   label='val')
ax1.set_title('Total Loss (L1 + edge)')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, history['pixel'], label='pixel L1')
ax2.plot(epochs, history['edge'],  label='edge (Sobel)')
ax2.set_title('Train Loss Components')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Visual sanity check — Clean vs Adversarial vs Denoised
import matplotlib.pyplot as plt

# Reload best checkpoint for visualization
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model.eval()

# Pick one example per attack type
attack_samples = {}
for clean_path, adv_path, attack_name in full_ds.pairs:
    if attack_name not in attack_samples:
        attack_samples[attack_name] = (clean_path, adv_path)
    if len(attack_samples) == 3:
        break

def bgr_to_display(bgr, max_side=384):
    h, w = bgr.shape[:2]
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        bgr   = cv2.resize(bgr, (int(w * scale), int(h * scale)))
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

n = len(attack_samples)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
if n == 1:
    axes = [axes]

for row, (attack_name, (clean_path, adv_path)) in enumerate(attack_samples.items()):
    clean_bgr = cv2.imread(clean_path)
    adv_bgr   = cv2.imread(adv_path)

    # Run model on full image (no crop)
    adv_rgb = cv2.cvtColor(adv_bgr, cv2.COLOR_BGR2RGB)
    adv_t   = torch.from_numpy(adv_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out_t = torch.clamp(model(adv_t, timestep=50.0), 0.0, 1.0)
    denoised_rgb = (out_t.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    denoised_bgr = cv2.cvtColor(denoised_rgb, cv2.COLOR_RGB2BGR)

    axes[row][0].imshow(bgr_to_display(clean_bgr))
    axes[row][0].set_title('Clean (target)')
    axes[row][0].axis('off')

    axes[row][1].imshow(bgr_to_display(adv_bgr))
    axes[row][1].set_title(f'Adversarial ({attack_name})')
    axes[row][1].axis('off')

    axes[row][2].imshow(bgr_to_display(denoised_bgr))
    axes[row][2].set_title('DPC-UNet Denoised')
    axes[row][2].axis('off')

plt.suptitle('Fine-tuned DPC-UNet: Adversarial Purification', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Next Steps

1. **Download** `dpc_unet_adversarial_finetuned.pt` from your Google Drive
2. **Copy** it into the `YOLO-Bad-Triangle/` project root
3. **Update** `.env`:
   ```
   DPC_UNET_CHECKPOINT_PATH=dpc_unet_adversarial_finetuned.pt
   ```
4. **Re-run the sweep** to evaluate the fine-tuned model:
   ```bash
   PYTHONPATH=src ./.venv/bin/python scripts/sweep_and_report.py \
     --attacks fgsm,pgd,deepfool,blur \
     --defenses c_dog,c_dog_ensemble,median_preprocess \
     --preset full \
     --workers 1 \
     --validation-enabled
   ```
5. **Compare** the new `framework_run_report.md` against the baseline sweep

### What to look for
- `c_dog` recovery should move from **−433%** (original) toward **positive** values
- Detection counts for defended runs should be closer to the attacked (not defended) baseline
- `deepfool + c_dog` is the hardest case — it should show the biggest relative improvement
  since deepfool was in the training data and caused the most damage in the original sweep

### If recovery is still poor
- Try increasing `NUM_EPOCHS` to 100
- Try reducing `TIMESTEP_MAX` to 50 (focus on the t=50 inference setting)
- Try increasing `EDGE_WEIGHT` to 0.3 to prioritize structural preservation more aggressively
- Consider adding a **YOLO feature loss**: run the YOLO backbone on `output` and `clean`,
  penalize L1 distance between intermediate feature maps — this directly optimizes for
  detection-relevant representations